# Ridge Regression (7 Features) on Walmart Weekly Sales - Technical Paper Version

Same format, hard-coded features: Store, Week, Holiday_Flag, Temperature, Fuel_Price, CPI, Unemployment.

In [ ]:
# 1) Imports and reproducibility
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
np.random.seed(42)

In [ ]:
# 2) Load data
csv_candidates = ['/Users/jaacabrera/Downloads/walmart-sales-dataset-of-45stores.csv','walmart-sales-dataset-of-45stores.csv']
df=None
for p in csv_candidates:
    try:
        df=pd.read_csv(p, low_memory=False); print(f'Loaded: {p}'); break
    except Exception: pass
if df is None: raise FileNotFoundError('Update dataset path.')

df['Date']=pd.to_datetime(df['Date'].astype('string'), errors='coerce', dayfirst=True)
df=df.dropna(subset=['Date']).reset_index(drop=True)
if 'Weekly_Sales' not in df.columns: raise KeyError('Weekly_Sales missing')
df=df.dropna(subset=['Weekly_Sales']).reset_index(drop=True)
print('Shape after checks:', df.shape); df.head(3)

In [ ]:
# 3) Hard-coded 7 features
if 'Store' not in df.columns: raise KeyError('Store missing')
df['Week']=df['Date'].dt.isocalendar().week.astype(int)
feature_cols=['Store','Week','Holiday_Flag','Temperature','Fuel_Price','CPI','Unemployment']
X=df[feature_cols].copy(); y=df['Weekly_Sales'].copy()
print('X:',X.shape,'y:',y.shape); print('Features:',feature_cols)

In [ ]:
# 4) 60/20/20 split
X_rem, X_test, y_rem, y_test = train_test_split(X,y,test_size=0.20,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rem,y_rem,train_size=0.25,random_state=42)
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)}")

In [ ]:
# 5) CV tuning (alpha) with scaling
imp=SimpleImputer(strategy='median')
X_train_imp=pd.DataFrame(imp.fit_transform(X_train),columns=feature_cols)
X_val_imp  =pd.DataFrame(imp.transform(X_val),columns=feature_cols)
X_test_imp =pd.DataFrame(imp.transform(X_test),columns=feature_cols)
sc=StandardScaler()
X_train_f=sc.fit_transform(X_train_imp); X_val_f=sc.transform(X_val_imp); X_test_f=sc.transform(X_test_imp)

param_grid={'alpha':[0.1,1.0,10.0]}
cv=GridSearchCV(Ridge(random_state=42),param_grid,cv=5,scoring='r2',n_jobs=-1,verbose=1)
cv.fit(X_train_f,y_train)
model=cv.best_estimator_; best=cv.best_params_; cv_r2=cv.best_score_
print('Best:',best,'CV R2:',round(cv_r2,4))

rmse=lambda yt,yp: float(np.sqrt(mean_squared_error(yt,yp)))
preds_train=model.predict(X_train_f); preds_val=model.predict(X_val_f); preds_test=model.predict(X_test_f)
metrics={
 'TRAIN':{'R2':r2_score(y_train,preds_train),'MSE':mean_squared_error(y_train,preds_train),'RMSE':rmse(y_train,preds_train),'MAE':mean_absolute_error(y_train,preds_train)},
 'VAL':  {'R2':r2_score(y_val,preds_val),'MSE':mean_squared_error(y_val,preds_val),'RMSE':rmse(y_val,preds_val),'MAE':mean_absolute_error(y_val,preds_val)},
 'TEST': {'R2':r2_score(y_test,preds_test),'MSE':mean_squared_error(y_test,preds_test),'RMSE':rmse(y_test,preds_test),'MAE':mean_absolute_error(y_test,preds_test)}
}
print('Test R2:',round(metrics['TEST']['R2'],4),'RMSE$',round(metrics['TEST']['RMSE'],2),'MAE$',round(metrics['TEST']['MAE'],2))

importances=pd.DataFrame({'Feature':feature_cols,'Importance':np.abs(model.coef_)}).sort_values('Importance',ascending=False)

# Exports
pd.DataFrame({'Dataset':['Train','Validation','Test'],
             'R²':[metrics['TRAIN']['R2'],metrics['VAL']['R2'],metrics['TEST']['R2']],
             'MSE':[metrics['TRAIN']['MSE'],metrics['VAL']['MSE'],metrics['TEST']['MSE']],
             'RMSE':[metrics['TRAIN']['RMSE'],metrics['VAL']['RMSE'],metrics['TEST']['RMSE']],
             'MAE':[metrics['TRAIN']['MAE'],metrics['VAL']['MAE'],metrics['TEST']['MAE']]}).to_csv('Ridge7F_Regression_Metrics.csv',index=False)

pd.DataFrame({'Hyperparameter':['alpha','cv_folds','cv_r2_score'],
             'Value':[best['alpha'],5,cv_r2],
             'Tuned':['Yes (GridSearchCV)','N/A','N/A'],
             'Description':['L2 regularization strength','Number of CV folds','Best R² across CV']}).to_csv('Ridge7F_Hyperparameters.csv',index=False)

importances.to_csv('Ridge7F_Feature_Importances.csv',index=False)

pd.DataFrame({'Actual':y_train.values,'Predicted':preds_train,'Residual':y_train.values-preds_train,
              'Residual_Squared':(y_train.values-preds_train)**2,'Absolute_Error':np.abs(y_train.values-preds_train)}).to_csv('Ridge7F_Train_Predictions.csv',index=False)
pd.DataFrame({'Actual':y_val.values,'Predicted':preds_val,'Residual':y_val.values-preds_val,
              'Residual_Squared':(y_val.values-preds_val)**2,'Absolute_Error':np.abs(y_val.values-preds_val)}).to_csv('Ridge7F_Validation_Predictions.csv',index=False)
pd.DataFrame({'Actual':y_test.values,'Predicted':preds_test,'Residual':y_test.values-preds_test,
              'Residual_Squared':(y_test.values-preds_test)**2,'Absolute_Error':np.abs(y_test.values-preds_test)}).to_csv('Ridge7F_Test_Predictions.csv',index=False)
print('✓ CSV exports written (metrics, hyperparameters, importances, predictions)')

In [ ]:
# 6) Parity Plot: Actual vs Predicted
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].scatter(y_train,preds_train,alpha=0.5,edgecolors='k',linewidth=0.5,color='green')
axes[0].plot([y_train.min(),y_train.max()],[y_train.min(),y_train.max()],'r--',lw=2); axes[0].set_title(f"Train (R²={metrics['TRAIN']['R2']:.4f})")
axes[1].scatter(y_val,preds_val,alpha=0.5,edgecolors='k',linewidth=0.5)
axes[1].plot([y_val.min(),y_val.max()],[y_val.min(),y_val.max()],'r--',lw=2); axes[1].set_title(f"Val (R²={metrics['VAL']['R2']:.4f})")
axes[2].scatter(y_test,preds_test,alpha=0.5,edgecolors='k',linewidth=0.5,color='orange')
axes[2].plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--',lw=2); axes[2].set_title(f"Test (R²={metrics['TEST']['R2']:.4f})")
for ax in axes:
    ax.set_xlabel('Actual Weekly Sales ($)'); ax.set_ylabel('Predicted Weekly Sales ($)'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

# 7) Model Configuration Summary & 8) Reporting
- Model: Ridge (sklearn.linear_model.Ridge)
- Features (7): Store, Week, Holiday_Flag, Temperature, Fuel_Price, CPI, Unemployment
- Split: 20% Train, 60% Val, 20% Test (random_state=42)
- Preprocessing: SimpleImputer(median) + StandardScaler
- Tuning (5-fold GridSearchCV): alpha ∈ {0.1, 1.0, 10.0}; scoring=R²
- CSVs: Ridge7F_Regression_Metrics.csv, Ridge7F_Hyperparameters.csv, Ridge7F_Feature_Importances.csv, Ridge7F_[Train|Validation|Test]_Predictions.csv